# 00 — Setup, data, and Phase-0 dry run

**Project:** RLVR plasticity pilot ("Seeing the Stall Coming", RQ1) · **Owner:** Aaron (Person 4)

Loads GSM8K + SVAMP, verifies fields, freezes all splits (probe set, train/eval
slices), unit-tests the metric & reward code, and dry-runs the Q metrics on the
base model with 8 prompts. Run once on CPU/T4 — costs <1 compute unit.

**Deviations from Tommy's spec logged in this notebook:** none (this notebook
makes no training choices). The probe-sensitivity superset tops up from a
held-out train slice because the GSM8K test split is too small for 2048
disjoint prompts — see `src/data.py`.

In [ ]:
%pip install -q trl==1.6.0 transformers==5.13.0 datasets==5.0.0 accelerate==1.14.0 pytest==8.4.2 numpy==2.3.5 scipy==1.16.3 pandas==2.3.3 matplotlib==3.10.6

In [ ]:
import json, os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/eaaj-pilot")  # upload the folder here
else:
    PROJECT_DIR = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print("project dir:", PROJECT_DIR)

## Datasets: load + verify fields (briefing §6 Phase 0)

In [ ]:
from src.data import load_gsm8k, load_svamp

gsm_train, gsm_test = load_gsm8k()
assert set(gsm_train.column_names) == {"question", "answer"}, gsm_train.column_names
assert "####" in gsm_train[0]["answer"], "GSM8K gold answers must carry '#### <num>'"
print(f"GSM8K ok: train={len(gsm_train)} test={len(gsm_test)}")

svamp = load_svamp()
for col in ("Body", "Question", "Answer"):
    assert col in svamp["train"].column_names, svamp["train"].column_names
print(f"SVAMP ok: train={len(svamp['train'])} test={len(svamp['test'])}")

## Freeze splits + probe set (committed to `data/`, seed 42)

Idempotent: if the committed JSONs already exist they are NOT regenerated —
the frozen probe set must stay identical across the whole experiment.

In [ ]:
from src.data import build_gsm8k_splits, build_svamp_splits, freeze_probe_set

if not Path("data/gsm8k_splits.json").exists():
    build_gsm8k_splits()
if not Path("data/probe_set_ids.json").exists():
    freeze_probe_set()
if not Path("data/svamp_splits.json").exists():
    build_svamp_splits()

probe = json.loads(Path("data/probe_set_ids.json").read_text())
splits = json.loads(Path("data/gsm8k_splits.json").read_text())
assert len(probe["probe_prompts"]) == 512 and len(probe["probe_big_prompts"]) == 2048
assert probe["probe_prompts"] == probe["probe_big_prompts"][:512]
assert not set(splits["gsm8k_eval_idx"]) & set(splits["probe_idx"])
print("splits frozen and invariants hold")

## Unit tests (metrics + reward parsing)

In [ ]:
ret = os.system(f"{sys.executable} -m pytest tests/ -q")
assert ret == 0, "unit tests failed — fix before spending any GPU units"

## Reward smoke test on real GSM8K rows

In [ ]:
from src.reward import exact_answer_reward, extract_gold_answer

rows = [gsm_train[i] for i in range(4)]
golds = [extract_gold_answer(r["answer"]) for r in rows]
print("golds:", golds)
# a fake correct completion and a fake wrong one per row
fake = [f"reasoning... #### {g}" for g in golds] + ["the answer is 999999"] * 4
r = exact_answer_reward(completions=fake, answer=golds + golds)
assert r == [1.0] * 4 + [0.0] * 4, r
print("reward function ok:", r)

## Q-metric dry run on the base model (8 prompts)

Verifies the full activation-collection path (hidden states + MLP
post-activation hooks) on Qwen2.5-0.5B. Values here are NOT comparable to the
real per-checkpoint measurements (n=8, not 512) — plumbing check only.

In [ ]:
from src.data import load_probe_prompts
from src.metrics import checkpoint_q_metrics

PILOT = json.loads(Path("pilot_config.json").read_text())
MODEL = PILOT["model_id"]
REVISION = PILOT["model_revision"]

tok = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
device = "mps" if torch.backends.mps.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(
    MODEL, revision=REVISION, dtype=torch.float32).to(device)

m = checkpoint_q_metrics(model, tok, load_probe_prompts()[:8], batch_size=4)
m.update({"model_id": MODEL, "model_revision": REVISION,
          "purpose": "plumbing_only_not_scientific_result"})
Path("outputs").mkdir(exist_ok=True)
Path("outputs/dry_run_metrics.json").write_text(json.dumps(m, indent=1))
print("dry run ok -> outputs/dry_run_metrics.json")